In [1]:
# ================================================
# PROJECT: Chicago Bulls Social Media & Performance
#          Time Series Analysis 2018-2023
# AUTHOR:  [Your Name]
# PURPOSE: Identify engagement spike drivers to guide
#          NBA sponsorship campaign timing decisions
# SKILLS:  Python, MySQL, Excel, Power BI, Time Series
# ================================================

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from nba_api.stats.endpoints import teamgamelog
from nba_api.stats.static import teams
import time
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully")

All libraries loaded successfully


In [3]:
# Get all NBA teams
nba_teams = teams.get_teams()

# Find Chicago Bulls specifically
bulls = [team for team in nba_teams if team['full_name'] == 'Chicago Bulls'][0]
bulls_id = bulls['id']

print(f"Chicago Bulls Team ID: {bulls_id}")
print(f"Abbreviation: {bulls['abbreviation']}")

Chicago Bulls Team ID: 1610612741
Abbreviation: CHI


In [6]:
seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23'] 
all_games = []

for season in seasons:
    print(f"Pulling data for {season} season...")
    gamelog = teamgamelog.TeamGameLog(
        team_id=bulls_id,
        season=season,
        season_type_all_star='Regular Season'
    )
    df_season = gamelog.get_data_frames()[0]
    df_season['SEASON'] = season
    all_games.append(df_season)
    time.sleep(1)  # pause 1 second between calls to avoid rate limiting

bulls_games = pd.concat(all_games, ignore_index=True)
print(f"\nTotal games pulled: {len(bulls_games)}")
print("Columns available:", bulls_games.columns.tolist())

Pulling data for 2018-19 season...
Pulling data for 2019-20 season...
Pulling data for 2020-21 season...
Pulling data for 2021-22 season...
Pulling data for 2022-23 season...

Total games pulled: 383
Columns available: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'SEASON']


In [7]:
# Convert date to proper date format
bulls_games['GAME_DATE'] = pd.to_datetime(bulls_games['GAME_DATE'])

# Sort by date oldest to newest
bulls_games = bulls_games.sort_values('GAME_DATE').reset_index(drop=True)

# Create win indicator (1 = win, 0 = loss)
bulls_games['WIN'] = (bulls_games['WL'] == 'W').astype(int)

# Create win streak column (consecutive wins)
bulls_games['WIN_STREAK'] = bulls_games['WIN'].groupby(
    (bulls_games['WIN'] != bulls_games['WIN'].shift()).cumsum()
).cumcount() + 1
bulls_games.loc[bulls_games['WIN'] == 0, 'WIN_STREAK'] = 0

# Create rolling 10-game win rate
bulls_games['WIN_RATE_10G'] = bulls_games['WIN'].rolling(10).mean().round(3)

# Flag COVID season
bulls_games['COVID_SEASON'] = (bulls_games['SEASON'] == '2019-20').astype(int)

print("Data cleaned successfully")
print(f"Date range: {bulls_games['GAME_DATE'].min()} to {bulls_games['GAME_DATE'].max()}")
print(f"Overall win rate: {bulls_games['WIN'].mean():.1%}")
print("\nWins per season:")
print(bulls_games.groupby('SEASON')['WIN'].sum())

Data cleaned successfully
Date range: 2018-10-18 00:00:00 to 2023-04-09 00:00:00
Overall win rate: 42.0%

Wins per season:
SEASON
2018-19    22
2019-20    22
2020-21    31
2021-22    46
2022-23    40
Name: WIN, dtype: int64


In [8]:
bulls_games.to_excel('bulls_game_data_2018_2023.xlsx', 
                       index=False, sheet_name='Game Logs')
print("Excel file saved: bulls_game_data_2018_2023.xlsx")

Excel file saved: bulls_game_data_2018_2023.xlsx
